# Data Preprocessing – BA Clustering
This notebook reads the raw data and transforms it into a DataFrame for cluster analysis.

In [1]:
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta

## 1. Load Data

In [ ]:
# Receipt line items (used for Recency, Frequency, and product group revenue)
df_receipts = pd.read_excel('data/Umsatz_nach_Konzern_Berichtsdaten.xlsx')
print(f'Receipt data: {len(df_receipts)} rows, {df_receipts["Adressen: Firma"].nunique()} companies')

# Total revenue per company (used for Monetary Value)
df_revenue = pd.read_csv('data/Umsatz_nach_Konzern.csv', sep=None, engine='python')
print(f'Revenue data: {len(df_revenue)} companies')
df_revenue.head(3)

In [3]:
# Product group definitions
df_product_groups = pd.read_excel('data/Produktgruppen.xlsx')
print(f'Product groups: {df_product_groups.shape}')
df_product_groups.head(3)

Product groups: (176, 8)


,Regale\n10XX - 18XX,Lager Zubehör\n22XX - 27XX + 39XX,Werkstätte\n28XX - 38XX,Werkstätten-Zubehör\n44XX - 48XX,Garderoben\n40XX,Garderoben-Zubehör\n41XX - 42XX,Büroeinrichtungen\n43XX + 50XX - 52XX,Sonstige Produkte
0,1001Z,2286A,2806K,4422A,4000Z,4098A,4322A,1900A
1,1025A,2293A,2825A,4422K,4001K,4099Z,4322K,1900B
2,1030A,2299A,2825B,4422Z,4004A,40AJL,4322Z,191AR


## 2. Build Product Group Mapping

Column headers in Produktgruppen.xlsx contain line breaks (`\n`).
The part before `\n` is the group name used in the final table.

In [4]:
# Mapping: product number prefix -> product group name
product_group_mapping = {}  # {prefix: group_name}
product_group_list = []

for col in df_product_groups.columns:
    # Group name = part before the first \n
    group_name = col.split('\n')[0].strip()
    product_group_list.append(group_name)
    for val in df_product_groups[col].dropna():
        prefix = str(val).strip()
        if prefix:
            product_group_mapping[prefix] = group_name

print(f'Product groups: {product_group_list}')
print(f'Mapped product numbers: {len(product_group_mapping)}')
print('Examples:', dict(list(product_group_mapping.items())[:5]))

Product groups: ['Regale', 'Lager Zubehör', 'Werkstätte', 'Werkstätten-Zubehör', 'Garderoben', 'Garderoben-Zubehör', 'Büroeinrichtungen', 'Sonstige Produkte']
Mapped product numbers: 734
Examples: {'1001Z': 'Regale', '1025A': 'Regale', '1030A': 'Regale', '1030B': 'Regale', '1030C': 'Regale'}


## 3. Preprocess Receipt Data

- Rows without a product number are removed
- The product prefix (part before `_`) is extracted
- Each row is mapped to a product group

In [ ]:
# Select relevant columns (including revenue per line item)
df_receipt_items = df_receipts[['Adressen: Firma', 'Belege: Datum', 'Belegpositionen: Produktnummer', 'Belegpositionen: Summe']].copy()
df_receipt_items.columns = ['company', 'date', 'product_number', 'revenue']

# Remove rows without a product number
n_before = len(df_receipt_items)
df_receipt_items = df_receipt_items.dropna(subset=['product_number'])
print(f'Rows before filter: {n_before}, after removing NaN product numbers: {len(df_receipt_items)}')

# Convert revenue to numeric (handles European format like "1 201,00" with space as thousands separator)
df_receipt_items['revenue'] = (
    df_receipt_items['revenue']
    .astype(str)
    .str.replace('\xa0', '', regex=False)   # non-breaking space
    .str.replace(' ', '', regex=False)       # regular space (thousands separator)
    .str.replace(',', '.', regex=False)      # decimal comma → point
    .pipe(pd.to_numeric, errors='coerce')
)

# Parse dates
df_receipt_items['date'] = pd.to_datetime(df_receipt_items['date'])

# Extract product prefix: part before "_"
df_receipt_items['product_prefix'] = df_receipt_items['product_number'].astype(str).str.split('_').str[0].str.strip()

# Map each product prefix to its product group
df_receipt_items['product_group'] = df_receipt_items['product_prefix'].map(product_group_mapping).fillna('Sonstige Produkte')

n_unmapped = df_receipt_items['product_group'].isna().sum()
print(f'Unmapped product numbers: {n_unmapped} ({n_unmapped / len(df_receipt_items) * 100:.1f}%)')
if n_unmapped > 0:
    print('Examples:', df_receipt_items[df_receipt_items['product_group'].isna()]['product_prefix'].value_counts().head(10).to_dict())

df_receipt_items.head()

In [6]:
df_receipt_items["product_group"].value_counts()

product_group
Regale                 4549
Werkstätte             4236
Sonstige Produkte      2530
Garderoben             2045
Garderoben-Zubehör     1522
Werkstätten-Zubehör     892
Lager Zubehör           504
Büroeinrichtungen       108
Name: count, dtype: int64

## 4. Calculate RFM Features

- **Recency**: date of the most recent receipt per company
- **Frequency**: number of unique purchase dates in the last 3 years per company
- **Product Group Revenue**: total revenue per product group per company (all time)

In [7]:
# Reference date for the "last 3 years" window
reference_date = pd.Timestamp.today().normalize()
cutoff_date = reference_date - pd.DateOffset(years=3)
print(f'Reference date: {reference_date.date()}, 3-year cutoff: {cutoff_date.date()}')

# --- Recency ---
# Most recent purchase date per company (using all receipt data)
df_purchase_dates = df_receipts[['Adressen: Firma', 'Belege: Datum']].copy()
df_purchase_dates.columns = ['company', 'date']
df_purchase_dates['date'] = pd.to_datetime(df_purchase_dates['date'])
recency = df_purchase_dates.groupby('company')['date'].max().rename('Recency')

# --- Frequency ---
# Number of unique purchase dates per company within the last 3 years
df_purchases_last_3_years = df_purchase_dates[df_purchase_dates['date'] >= cutoff_date]
frequency = (
    df_purchases_last_3_years.groupby('company')['date']
    .nunique()
    .rename('Frequency')
)

print(f'Recency: {len(recency)} companies')
print(f'Frequency: {len(frequency)} companies with purchases in last 3 years')

Reference date: 2026-04-12, 3-year cutoff: 2023-04-12
Recency: 615 companies
Frequency: 319 companies with purchases in last 3 years


In [ ]:
# --- Product Group Revenue ---
# Only use rows that were successfully mapped to a product group
df_receipt_items_mapped = df_receipt_items.dropna(subset=['product_group'])

# Sum revenue per company and product group (instead of counting purchases)
pg_revenue = (
    df_receipt_items_mapped
    .groupby(['company', 'product_group'])['revenue']
    .sum()
    .unstack(fill_value=0)
)

# Ensure all product groups exist as columns (even if no company bought from them)
for group in product_group_list:
    if group not in pg_revenue.columns:
        pg_revenue[group] = 0

# Reorder columns to match the defined group order
pg_revenue = pg_revenue[product_group_list]

print('Product group revenue shape:', pg_revenue.shape)
pg_revenue.head()

## 5. Group receipts by companies

In [ ]:
# Start with all companies from the receipts dataset
all_companies = df_receipts['Adressen: Firma'].dropna().unique()
df_companies = pd.DataFrame({'ID': all_companies}).set_index('ID')

# Join Recency and Frequency
df_companies = df_companies.join(recency.rename_axis('ID'), how='left')
df_companies = df_companies.join(frequency.rename_axis('ID'), how='left')
df_companies['Frequency'] = df_companies['Frequency'].fillna(0).astype(int)

# Join Monetary Value (total revenue per company over all years)
total_revenue = df_revenue.set_index('Firma')['Ergebnis']
df_companies = df_companies.join(total_revenue.rename_axis('ID').rename('Monetary Value'), how='left')

# Join product group revenue
df_companies = df_companies.join(pg_revenue.rename_axis('ID'), how='left')
df_companies[product_group_list] = df_companies[product_group_list].fillna(0)

# Reset index
df_companies = df_companies.reset_index()

print(f'Final table: {df_companies.shape}')
df_companies.head(10)

In [10]:
# Overview
print('Columns:', df_companies.columns.tolist())
print()
print(df_companies.describe())

Columns: ['ID', 'Recency', 'Frequency', 'Monetary Value', 'Regale', 'Lager Zubehör', 'Werkstätte', 'Werkstätten-Zubehör', 'Garderoben', 'Garderoben-Zubehör', 'Büroeinrichtungen', 'Sonstige Produkte']

                             Recency   Frequency  Monetary Value  \
count                            615  615.000000    6.150000e+02   
mean   2023-04-19 07:46:19.534959360    2.754472    2.761275e+04   
min              2019-09-10 00:00:00    0.000000    0.000000e+00   
25%              2021-05-02 03:53:24    0.000000    5.121000e+02   
50%              2023-06-30 11:42:23    1.000000    2.085000e+03   
75%              2025-04-07 02:15:45    2.000000    1.358835e+04   
max              2026-03-13 13:23:36  402.000000    2.361920e+06   
std                              NaN   17.253613    1.189219e+05   

              Regale  Lager Zubehör    Werkstätte  Werkstätten-Zubehör  \
count     615.000000     615.000000  6.150000e+02           615.000000   
mean     7120.527496     363.032520  6

## 6. Company Enrichment – Bundesland, Industry, Company Size

Load `firmen.csv`, filter to main companies only (`S4_ANZAHLMUTTER = 0`, `GWISCOMPANY = True`, `GWISCONTACT = False`),
keep Bundesland (`GWSTATE1`), industry, and company size, then join onto `df_final`.
Companies without a Bundesland are removed (saved to `companies_without_location.csv`).
Missing company sizes are inferred from the company name using legal form indicators.

In [ ]:
# Load Alle_Firmen.csv (company sizes already pre-filled in fake_company_size.ipynb)
df_firmen_original = pd.read_csv('data/Alle_Firmen.csv', sep=None, engine='python')

# Normalize bool columns that Excel may have saved as strings ('True'/'False')
for col in ['GWISCOMPANY', 'GWISCONTACT']:
    df_firmen_original[col] = df_firmen_original[col].map(
        lambda x: str(x).strip().lower() == 'true' if pd.notna(x) else False
    )

# Filter: main companies only (no subsidiaries, no contacts)
df_firmen = df_firmen_original[
    (df_firmen_original['S4_ANZAHLMUTTER'] == 0) &
    (df_firmen_original['GWISCOMPANY'] == True) &
    (df_firmen_original['GWISCONTACT'] == False)
].copy()

# Keep only relevant columns: company name, Bundesland, industry, company size
df_firmen = df_firmen[['COMPNAME', 'GWSTATE1', 'GWBRANCH', 'OS_MA_ANZ_GRUP']].copy()
df_firmen.columns = ['company', 'bundesland', 'industry', 'company_size']


print(f'Companies after filter: {len(df_firmen)}')
print(f'\nBundesland distribution (top 10):')
print(df_firmen['bundesland'].value_counts(dropna=False).head(10).to_string())
print(f'\ncompany_size distribution:')
print(df_firmen['company_size'].value_counts(dropna=False).to_string())
df_firmen.head()


In [ ]:
# Map Bundesland, industry and company size onto df_companies using company name as key
df_firmen_lookup = df_firmen.drop_duplicates(subset='company').set_index('company')

df_companies = df_companies.join(
    df_firmen_lookup[['bundesland', 'industry', 'company_size']],
    on='ID', how='left'
)

n_matched = df_companies['bundesland'].notna().sum()
print(f'Companies matched with firmen data: {n_matched} / {len(df_companies)} ({n_matched / len(df_companies) * 100:.1f}%)')

# Save companies without Bundesland before removing them
df_companies_without_location = df_companies[df_companies['bundesland'].isna()]
#df_companies_without_location.to_csv('data/companies_without_location.csv', index=False)
print(f'Companies without Bundesland (removed & saved): {len(df_companies_without_location)}')

df_companies = df_companies.dropna(subset=['bundesland']).reset_index(drop=True)
print(f'Remaining companies: {len(df_companies)}')
df_companies.head()

In [13]:
# Company sizes were already pre-filled in fake_company_size.ipynb
print(f'Companies with missing company size: {df_companies["company_size"].isna().sum()}')
print(f'\ncompany_size distribution:')
print(df_companies['company_size'].value_counts(dropna=False).to_string())

Companies with missing company size: 0

company_size distribution:
company_size
11-50 Kleinunternehmen            214
51-350 Mittelunternehmen          197
1-10 Kleinstunternehmen            72
351-1000 Mittelgroßunternehmen     48
>1000 Großunternehmen              25


In [ ]:
df_companies.head()

In [15]:
df_companies.to_csv('data/df_final_before_encoding_and_scaling.csv', index=False)

## 7. Feature Engineering for Clustering

- `industry_group` is simplified into merged groups (ZG 1+2, ZG 3+4, ZG 5+6, ZG 7, ZG 8, ZG 9); rows with `Sonstige` are removed  
- Product groups are merged: Werkstätte + Werkstätten-Zubehör → `Werkstätte_gesamt`, Garderoben + Garderoben-Zubehör → `Garderoben_gesamt`, Büroeinrichtungen → `Sonstige Produkte`  
- Company sizes are simplified: Klein, Mittel, Groß  
- Bundesländer are grouped into regions: Ost, Süd, West  
- `Recency` is converted to days since the most recent purchase  
- Company names are replaced by a numeric `customer_id`  
- Categorical columns are one-hot encoded  
- All numeric features are standardised with z-score (StandardScaler)

In [16]:
print(df_companies['company_size'].value_counts(dropna=False).to_string())

company_size
11-50 Kleinunternehmen            214
51-350 Mittelunternehmen          197
1-10 Kleinstunternehmen            72
351-1000 Mittelgroßunternehmen     48
>1000 Großunternehmen              25


In [17]:
print(df_companies.isna().sum())

ID                     0
Recency                0
Frequency              0
Monetary Value         0
Regale                 0
Lager Zubehör          0
Werkstätte             0
Werkstätten-Zubehör    0
Garderoben             0
Garderoben-Zubehör     0
Büroeinrichtungen      0
Sonstige Produkte      0
bundesland             0
industry               0
company_size           0
dtype: int64


In [18]:
print(df_companies['industry'].value_counts().to_string())

industry
ZG 7 öffentl. Verwaltung - Versorger - Entsorger - Gesundheit - Schulen - Kultur                                                                                                                                                            62
ZG 7 öffentl. Verwaltung - Versorger - Entsorger - Gesundheit - Schulen - Kultur | Bildung Personen | Universitäten und Hochschulen                                                                                                         23
ZG 7 öffentl. Verwaltung - Versorger - Entsorger - Gesundheit - Schulen - Kultur | Bildung Personen | allgem. Schulen                                                                                                                       20
ZG 7 öffentl. Verwaltung - Versorger - Entsorger - Gesundheit - Schulen - Kultur | öffentl. Verwaltung | Stadt- und Gemeindeverwaltung                                                                                                      17
ZG 9 Tourismus - Gastronomie und Sp

In [19]:
import re

def map_to_industry_group(industry):
    """Maps raw industry strings directly to the final reduced industry groups."""
    if pd.isna(industry):
        return 'Sonstige'
    
    # Extract all ZG numbers mentioned
    zg_numbers = re.findall(r'ZG\s*(\d+)', str(industry))
    
    # Filter out 99
    valid_zg = [int(n) for n in zg_numbers if n != '99']
    
    if not valid_zg:
        return 'Sonstige'
    
    first_zg = valid_zg[0]
    
    # Map directly to final groups
    if first_zg in (1, 2):
        return 'ZG 1+2'
    elif first_zg in (3, 4):
        return 'ZG 3+4'
    elif first_zg in (5, 6):
        return 'ZG 5+6'
    elif first_zg == 7:
        return 'ZG 7'
    elif first_zg == 8:
        return 'ZG 8'
    elif first_zg == 9:
        return 'ZG 9'
    else:
        return 'Sonstige'

df_companies['industry_group'] = df_companies['industry'].apply(map_to_industry_group)

print('Industry groups (final):')
print(df_companies['industry_group'].value_counts().to_string())

Industry groups (final):
industry_group
ZG 7        247
ZG 1+2      112
ZG 3+4       55
ZG 5+6       49
ZG 8         47
ZG 9         38
Sonstige      8


In [20]:
print(df_companies["bundesland"].value_counts())

bundesland
Wien                 254
Niederösterreich     154
Steiermark            46
Oberösterreich        28
Burgenland            24
Salzburg              17
Tirol                 11
Kärnten               10
Vorarlberg             5
Hessen                 2
Aargau                 2
Podravska              1
Baden-Württemberg      1
Stredoceský kraj       1
Name: count, dtype: int64


In [ ]:
# Map bundesland to regions (Ost / Süd / West)

bundesland_to_region_mapping = {
    'Wien': 'Ost',
    'Niederösterreich': 'Ost',
    'Burgenland': 'Ost',
    'Kärnten': 'Süd',
    'Steiermark': 'Süd',
    'Oberösterreich': 'West',
    'Salzburg': 'West',
    'Tirol': 'West',
    'Vorarlberg': 'West',
}

# Everything not in the mapping (foreign countries etc.) becomes 'West'
df_companies['bundesland'] = df_companies['bundesland'].map(bundesland_to_region_mapping).fillna('West')

print('Bundesland regions (final):')
print(df_companies['bundesland'].value_counts().to_string())

Bundesland regions (final):
bundesland
Ost     432
Süd      73
West     51


In [22]:
# --- Save raw df_companies before any transformation ---
df_companies.to_csv('data/df_final_saved_before_scaling_and_encoding.csv', index=False)
print(f'Saved raw df_companies: {df_companies.shape}')
print(f'Columns: {df_companies.columns.tolist()}')

Saved raw df_companies: (556, 16)
Columns: ['ID', 'Recency', 'Frequency', 'Monetary Value', 'Regale', 'Lager Zubehör', 'Werkstätte', 'Werkstätten-Zubehör', 'Garderoben', 'Garderoben-Zubehör', 'Büroeinrichtungen', 'Sonstige Produkte', 'bundesland', 'industry', 'company_size', 'industry_group']


In [23]:
# --- Recency: Datum → Anzahl Tage seit letztem Kauf ---
reference_date = pd.Timestamp.today().normalize()
df_companies['Recency'] = (reference_date - pd.to_datetime(df_companies['Recency'])).dt.days
print(f'Recency (days) – min: {df_companies["Recency"].min()}, max: {df_companies["Recency"].max()}, mean: {df_companies["Recency"].mean():.1f}')

# --- Mapping speichern: Index (customer_id) → Firmenname ---
df_id_mapping = df_companies[['ID']].copy()
df_id_mapping.index.name = 'customer_id'
df_id_mapping = df_id_mapping.rename(columns={'ID': 'company_name'})
df_id_mapping.to_csv('data/customer_id_mapping.csv')

# Firmenname entfernen – der Index dient als customer_id
df_companies = df_companies.drop(columns=['ID'])

print(f'\nCustomer ID mapping saved ({len(df_id_mapping)} rows).')
df_companies.head(3)

Recency (days) – min: 29, max: 2406, mean: 1057.7

Customer ID mapping saved (556 rows).


,Recency,Frequency,Monetary Value,Regale,Lager Zubehör,Werkstätte,Werkstätten-Zubehör,Garderoben,Garderoben-Zubehör,Büroeinrichtungen,Sonstige Produkte,bundesland,industry,company_size,industry_group
0,144,3,22610.0,19250.0,0.0,0.0,3360.0,0.0,0.0,0.0,0.0,West,ZG 1 Industrie/Produktion mit Betriebseinricht...,351-1000 Mittelgroßunternehmen,ZG 1+2
1,2346,0,149.0,0.0,0.0,0.0,0.0,149.0,0.0,0.0,0.0,Ost,ZG 7 öffentl. Verwaltung - Versorger - Entsorg...,11-50 Kleinunternehmen,ZG 7
2,169,3,10365.8,1076.7,0.0,6811.0,690.2,1629.0,139.9,0.0,19.0,Ost,ZG 2 Industrie/Produktion mit Betriebseinrichu...,51-350 Mittelunternehmen,ZG 1+2


In [24]:
# --- Dimension Reduction & Encoding ---

# Drop raw industry column (keep industry_group which is already in final groups)
df_companies = df_companies.drop(columns=['industry'])

# === 1. Merge product groups ===
df_companies['Werkstätte_gesamt'] = df_companies['Werkstätte'] + df_companies['Werkstätten-Zubehör']
df_companies['Garderoben_gesamt'] = df_companies['Garderoben'] + df_companies['Garderoben-Zubehör']
df_companies['Sonstige Produkte'] = df_companies['Sonstige Produkte'] + df_companies['Büroeinrichtungen']

df_companies = df_companies.drop(columns=[
    'Werkstätte', 'Werkstätten-Zubehör',
    'Garderoben', 'Garderoben-Zubehör',
    'Büroeinrichtungen'
])

product_group_columns = ['Regale', 'Lager Zubehör', 'Werkstätte_gesamt', 'Garderoben_gesamt', 'Sonstige Produkte']
print('=== Product groups after merging ===')
print(f'Columns: {product_group_columns}')
print(df_companies[product_group_columns].describe().round(1))

# === 2. Merge company size ===
company_size_mapping = {
    '1-10 Kleinstunternehmen': 'Klein',
    '11-50 Kleinunternehmen': 'Klein',
    '51-350 Mittelunternehmen': 'Mittel',
    '351-1000 Mittelgroßunternehmen': 'Groß',
    '>1000 Großunternehmen': 'Groß',
}
df_companies['company_size'] = df_companies['company_size'].map(company_size_mapping)
print('\n=== Company size after merging ===')
print(df_companies['company_size'].value_counts().to_string())

# === 3. Remove rows with "Sonstige" industry ===
n_before_sonstige_removal = len(df_companies)
df_companies = df_companies.dropna(subset=['industry_group'])
df_companies = df_companies[df_companies['industry_group'] != 'Sonstige']
n_removed_sonstige = n_before_sonstige_removal - len(df_companies)
print(f'\n=== Removed {n_removed_sonstige} rows with "Sonstige" industry ===')
print(f'Remaining companies: {len(df_companies)}')
print(df_companies['industry_group'].value_counts().to_string())

# === 4. Update customer ID mapping (without Sonstige) ===
df_original_mapping = pd.read_csv('data/customer_id_mapping.csv', index_col='customer_id')
df_id_mapping_final = df_original_mapping.loc[df_companies.index]
df_id_mapping_final.to_csv('data/customer_id_mapping.csv')
print(f'\n=== Updated customer ID mapping: {len(df_id_mapping_final)} companies ===')

# === 5. One-Hot Encoding ===
categorical_columns = ['industry_group', 'bundesland', 'company_size']
df_companies = pd.get_dummies(df_companies, columns=categorical_columns, dtype=int)

print(f'\n=== Final shape after one-hot encoding: {df_companies.shape} ===')
print(f'Columns: {df_companies.columns.tolist()}')

# === 6. Save unscaled data (for cluster interpretation later) ===
df_companies.to_csv('data/df_final_before_scaler.csv', index_label='customer_id')
print(f'\nSaved df_final_before_scaler.csv: {df_companies.shape}')

=== Product groups after merging ===
Columns: ['Regale', 'Lager Zubehör', 'Werkstätte_gesamt', 'Garderoben_gesamt', 'Sonstige Produkte']
         Regale  Lager Zubehör  Werkstätte_gesamt  Garderoben_gesamt  \
count     556.0          556.0              556.0              556.0   
mean     7655.0          399.9             8314.8            10914.8   
std     24409.4         2089.8            95509.7            46175.6   
min         0.0            0.0                0.0                0.0   
25%         0.0            0.0                0.0                0.0   
50%         0.0            0.0                0.0                0.0   
75%      2382.5            0.0              264.4             2473.5   
max    284514.6        40326.6          2215815.4           588404.6   

       Sonstige Produkte  
count              556.0  
mean              2849.9  
std               8358.5  
min              -1632.0  
25%                  0.0  
50%                 29.5  
75%               1462.8 

In [25]:
from sklearn.preprocessing import StandardScaler

# Alle numerischen Spalten skalieren
numeric_cols = df_companies.select_dtypes(include='number').columns.tolist()

scaler = StandardScaler()
df_final = df_companies.copy()
df_final[numeric_cols] = scaler.fit_transform(df_companies[numeric_cols])

print(f'Scaled {len(numeric_cols)} numeric columns.')
print(df_final[numeric_cols].describe().round(3))
df_final.head(3)

Scaled 20 numeric columns.
       Recency  Frequency  Monetary Value   Regale  Lager Zubehör  \
count  548.000    548.000         548.000  548.000        548.000   
mean     0.000      0.000          -0.000   -0.000          0.000   
std      1.001      1.001           1.001    1.001          1.001   
min     -1.366     -0.164          -0.239   -0.312         -0.193   
25%     -0.948     -0.164          -0.235   -0.312         -0.193   
50%     -0.183     -0.109          -0.221   -0.312         -0.193   
75%      0.898     -0.054          -0.124   -0.213         -0.193   
max      1.790     21.889          18.590   11.445         18.986   

       Sonstige Produkte  Werkstätte_gesamt  Garderoben_gesamt  \
count            548.000            548.000            548.000   
mean              -0.000              0.000             -0.000   
std                1.001              1.001              1.001   
min               -0.532             -0.087             -0.238   
25%               -0.

,Recency,Frequency,Monetary Value,Regale,Lager Zubehör,Sonstige Produkte,Werkstätte_gesamt,Garderoben_gesamt,industry_group_ZG 1+2,industry_group_ZG 3+4,industry_group_ZG 5+6,industry_group_ZG 7,industry_group_ZG 8,industry_group_ZG 9,bundesland_Ost,bundesland_Süd,bundesland_West,company_size_Groß,company_size_Klein,company_size_Mittel
0,-1.213131,0.001001,-0.059224,0.483673,-0.19268,-0.337673,-0.052175,-0.237917,1.973032,-0.334009,-0.313363,-0.905869,-0.306288,-0.272965,-1.85884,-0.392026,3.155947,2.591970,-1.025882,-0.746203
1,1.710647,-0.163568,-0.238288,-0.311759,-0.19268,-0.337673,-0.087135,-0.234709,-0.506834,-0.334009,-0.313363,1.103913,-0.306288,-0.272965,0.53797,-0.392026,-0.316862,-0.385807,0.974771,-0.746203
2,-1.179936,0.001001,-0.156837,-0.267269,-0.19268,-0.335407,-0.009088,-0.199838,1.973032,-0.334009,-0.313363,-0.905869,-0.306288,-0.272965,0.53797,-0.392026,-0.316862,-0.385807,-1.025882,1.340119


In [26]:
# --- Finalen Datensatz speichern ---
df_final.to_csv('data/df_final.csv', index_label='customer_id')
print(f'Saved df_final: {df_final.shape}')
print('Columns:', df_final.columns.tolist())

Saved df_final: (548, 20)
Columns: ['Recency', 'Frequency', 'Monetary Value', 'Regale', 'Lager Zubehör', 'Sonstige Produkte', 'Werkstätte_gesamt', 'Garderoben_gesamt', 'industry_group_ZG 1+2', 'industry_group_ZG 3+4', 'industry_group_ZG 5+6', 'industry_group_ZG 7', 'industry_group_ZG 8', 'industry_group_ZG 9', 'bundesland_Ost', 'bundesland_Süd', 'bundesland_West', 'company_size_Groß', 'company_size_Klein', 'company_size_Mittel']
